In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

#importing word cloud for text visualization
from wordcloud import WordCloud

# importing nltk for natural language processing
import nltk
from nltk.corpus import stopwords  #for stopword

# downloading nltk data
nltk.download('stopwords')       #downloading stopword data
nltk.download('punkt')          #downloading tokenizer data

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\CoreX\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\CoreX\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.


True

In [3]:
%pip install wordcloud

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [5]:
# read the csv file
df=pd.read_csv('spam.csv')

# Displaying first rows of the dataframe
df.head(10)

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN
5,spam,FreeMsg Hey there darling it's been 3 week's n...,NaN,NaN,NaN
6,ham,Even my brother is not like to speak with me. ...,NaN,NaN,NaN
7,ham,As per your request 'Melle Melle (Oru Minnamin...,NaN,NaN,NaN
8,spam,WINNER!! As a valued network customer you have...,NaN,NaN,NaN
9,spam,Had your mobile 11 months or more? U R entitle...,NaN,NaN,NaN


In [12]:
df.drop(columns = ['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], axis=1, inplace=True)
df.head()

KeyError: "['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'] not found in axis"

In [13]:
df.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [14]:
# Rename the columns name

df.rename(columns = {'v1': 'target', 'v2': 'text'}, inplace=True)

In [15]:
df.head()

,target,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


## Data Preprocessing

In [16]:
from sklearn.preprocessing import LabelEncoder

encoder=LabelEncoder()
df['target']=encoder.fit_transform(df['target'])

df.head()

,target,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [17]:
# check duplicate values in the dataset
df.duplicated().sum()

np.int64(403)

In [18]:
len(df)

5572

In [19]:
# remove duplicate values from the dataset
df=df.drop_duplicates(keep='first')
len(df)

5169

## Feature engineering

In [21]:
# importing the porter stemer for text stemming
from nltk.stem.porter import PorterStemmer

# importing the string module for handelling special characters
import string

# creating the instance of the porter stemmer
ps = PorterStemmer()

In [23]:
# lowercase transformation and text preprocessing function
def transform_text(text):
    # transform text into lowercase
    text=text.lower()

    #tokenization using nltk
    text=nltk.word_tokenize(text)

    #removing special characters and punctuation
    y=[]
    for i in text:
        if i.isalnum():
            y.append(i)

    # removing the stopword and punctuation 
    text=y[:]
    y.clear()

    # loop through the tokens and remove stopword and punctuation
    for i in text:
        if i not in stopwords.words('english') and i not in string.punctuation:
            y.append(i)

    # stemming the text using the porter stemmer
    text=y[:]
    y.clear()
    for i in text:
        y.append(ps.stem(i))

    # join the processed tokens back into a single string
    return " ".join(y)

In [29]:
import nltk
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\CoreX\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [30]:
transform_text('Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...')

'go jurong point crazi avail bugi n great world la e buffet cine got amor wat'

In [31]:
df['transformed_text']=df['text'].apply(transform_text)
df.head()

C:\Users\CoreX\AppData\Local\Temp\ipykernel_11348\2799170265.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['transformed_text']=df['text'].apply(transform_text)


,target,text,transformed_text
0,0,"Go until jurong point, crazy.. Available only ...",go jurong point crazi avail bugi n great world...
1,0,Ok lar... Joking wif u oni...,ok lar joke wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entri 2 wkli comp win fa cup final tkt 21...
3,0,U dun say so early hor... U c already then say...,u dun say earli hor u c alreadi say
4,0,"Nah I don't think he goes to usf, he lives aro...",nah think goe usf live around though


In [32]:
from sklearn.feature_extraction.text import CountVectorizer , TfidfVectorizer
tfid= TfidfVectorizer(max_features=3000)

In [33]:
x=tfid.fit_transform(df['transformed_text']).toarray()
y=df['target'].values

# Train Test Split

In [34]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test= train_test_split(x,y,test_size=0.2, random_state=2)

# model training

In [37]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from xgboost import XGBClassifier

In [38]:
svc = SVC(kernel= "sigmoid", gamma  = 1.0)
knc = KNeighborsClassifier()
mnb = MultinomialNB()
dtc = DecisionTreeClassifier(max_depth = 5)
lrc = LogisticRegression(solver = 'liblinear', penalty = 'l1')
rfc = RandomForestClassifier(n_estimators = 50, random_state = 2 )
abc = AdaBoostClassifier(n_estimators = 50, random_state = 2)
bc = BaggingClassifier(n_estimators = 50, random_state = 2)
etc = ExtraTreesClassifier(n_estimators = 50, random_state = 2)
gbdt = GradientBoostingClassifier(n_estimators = 50, random_state = 2)    
xgb  = XGBClassifier(n_estimators = 50, random_state = 2)

In [39]:
clfs = {
    'SVC': svc,
    'KNN': knc,
    'NB': mnb,
    'DT': dtc,
    'LR': lrc,
    'RF': rfc,
    'Adaboost': abc,
    'Bgc': bc,
    'ETC': etc,
    'GBDT': gbdt,
    'xgb': xgb
    
}

# Model evaluation

In [40]:
from sklearn.metrics import accuracy_score, precision_score

def train_classifier(clf, x_train, y_train, x_test, y_test):
    # fitting the model
    clf.fit(x_train,y_train)
    #making the prediction
    y_pred=clf.predict(x_test)
    #calculating the accuracy
    accuracy=accuracy_score(y_test,y_pred)
    precision=precision_score(y_test,y_pred)
    return accuracy, precision

In [41]:
accuracy_scores = []
precision_scores = []
for name, clfs in clfs.items():
    current_accuracy, current_precision = train_classifier(clfs, x_train, y_train, x_test, y_test)
    print()
    print("for:",name)
    print("accuracy:", current_accuracy)
    print("precision:", current_precision)

    accuracy_scores.append(current_accuracy)
    precision_scores.append(current_precision)


for: SVC
accuracy: 0.9758220502901354
precision: 0.9747899159663865

for: KNN
accuracy: 0.9052224371373307
precision: 1.0

for: NB
accuracy: 0.9709864603481625
precision: 1.0

for: DT
accuracy: 0.9323017408123792
precision: 0.8333333333333334

for: LR
accuracy: 0.9584139264990329
precision: 0.9702970297029703

for: RF
accuracy: 0.9758220502901354
precision: 0.9829059829059829

for: Adaboost
accuracy: 0.9245647969052224
precision: 0.8488372093023255

for: Bgc
accuracy: 0.9584139264990329
precision: 0.8682170542635659

for: ETC
accuracy: 0.9748549323017408
precision: 0.9745762711864406

for: GBDT
accuracy: 0.9468085106382979
precision: 0.9191919191919192

for: xgb
accuracy: 0.9700193423597679
precision: 0.9495798319327731
